# Per-seed threshold test

> "Line 669 states that the validation set was not used for labeling, and the response states that labeling was performed independently for each seed; however, the labels for every seed use the thresholds fitted to the seed-42 training set (lines 680-684), which contains part of the test and validation sets of the other nine seeds. The per-seed-threshold results stored in the shared code (sensitivity_results.pkl and reg_sensitivity_results.pkl) differ little from Table 1 ... Please correct the statement at line 669 and report the per-seed results in the Supplementary Information."

In the main pipeline (notebooks 2–6), the Optuna and K-means thresholds fitted on the **seed-42 training set** are applied to all 10 seeds.
This notebook repeats the analysis with **each seed's own thresholds**, fitted on that seed's training set only
(`threshold_log.pkl`, `kmeans_threshold_log.pkl`, both produced by `2. labeling.ipynb`), so that no seed's test or validation samples influence its own labels.

The main pipeline is not changed by this notebook.

| Analysis | Scope |
|---|---|
| Feature selection | 4 conditions × 10 seeds. Fold-change Top10 is computed from day-wise CLR means and does not depend on labels; VIP ≥ 1.0 is re-selected with the per-seed labels |
| CV test performance | 4 conditions × 5 models × 10 seeds, classification and regression |

The seed-42 fixed-threshold results are read from the saved outputs of notebooks 4 and 5; only the per-seed-threshold models are trained here.
Differences between the two schemes are tested with a paired t-test (pairs = seeds, n = 10).

**Run after notebooks 1–5** (this notebook reads their saved outputs).
Outputs: `output/pkl/per_seed_threshold_results.pkl`, `output/per_seed_threshold_test.xlsx`

In [1]:
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── TensorFlow 로그 억제 (5. regression.ipynb 와 동일) ─────────────
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
_fd, _saved = os.open(os.devnull, os.O_WRONLY), os.dup(2)
os.dup2(_fd, 2)
try:
    import tensorflow as tf
finally:
    os.dup2(_saved, 2); os.close(_saved); os.close(_fd)
tf.get_logger().setLevel('ERROR')
try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
except ImportError:
    pass

import sklearn.base as _base
from scipy import stats
from scipy.stats import f_oneway
from statsmodels.stats.multitest import multipletests
from sklearn.cross_decomposition import PLSRegression
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMRegressor

# Repository-relative paths. Run this notebook from the repository root.
OUTPUT_DIR = "output"
PKL_DIR    = f"{OUTPUT_DIR}/pkl"

SEEDS       = [38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
LABEL_SEED  = 42
LABEL_ORDER = [0, 2, 1]   # Early → Intermediate → Late

CONDS = ['cond1', 'cond2', 'cond3', 'cond4']
COND_LABELS = {
    'cond1': 'Optuna × FC Top10',
    'cond2': 'Optuna × VIP≥1.0',
    'cond3': 'K-means × FC Top10',
    'cond4': 'K-means × VIP≥1.0',
}
COND_SCREEN_KEY = {'cond1': 'cond1_optuna_fc10', 'cond2': 'cond2_optuna_vip',
                   'cond3': 'cond3_kmeans_fc10', 'cond4': 'cond4_kmeans_vip'}
COND_LABELING   = {'cond1': 'optuna', 'cond2': 'optuna',
                   'cond3': 'kmeans', 'cond4': 'kmeans'}

FINAL_COND, FINAL_MODEL = 'cond1', 'RandomForest'

## 1. Functions and model settings

Copied without modification from the main pipeline so that the per-seed analysis uses exactly the same procedures:

| Object | Source |
|---|---|
| `label_by_tpc` | `2. labeling.ipynb` |
| `extract_day`, `anova_fdr`, `vip_select`, `fold_change_top10` | `3. screening.ipynb` |
| `clf_configs` | `4. classification.ipynb` (`model_configs`) |
| `LSTMRegressor`, `reg_configs` | `5. regression.ipynb` (`model_configs`) |

Section 3 checks that these copies reproduce `screening_results.pkl` under the seed-42 thresholds before any model is trained.

In [2]:
def extract_day(desc):
    m = re.match(r'A(\d+)-\d+', str(desc))
    return int(m.group(1)) if m else np.nan


def label_by_tpc(df, tpc_thr_low, tpc_thr_high):
    """Early=0, Late=1, Intermediate=2"""
    cond_fresh = df[tpc_col] < tpc_thr_low
    cond_decay = df[tpc_col] >= tpc_thr_high
    return np.where(cond_fresh, 0, np.where(cond_decay, 1, 2))


def anova_fdr(df_train, feature_cols, alpha=0.05):
    """Day-based one-way ANOVA + BH FDR (q < alpha). Train only."""
    df = df_train.copy()
    if 'Day' not in df.columns:
        df['Day'] = df['Description'].apply(extract_day)

    p_values, feats_valid = [], []
    for feat in feature_cols:
        groups = [g[feat].dropna().values for _, g in df.groupby('Day')]
        valid  = [g for g in groups if len(g) > 1]
        if len(valid) >= 2:
            _, p = f_oneway(*valid)
            p_values.append(p)
            feats_valid.append(feat)

    if not p_values:
        return [], pd.DataFrame(columns=['feature', 'p_value', 'q_value', 'significant'])

    p_arr = np.array(p_values)
    reject, q_arr, _, _ = multipletests(p_arr, method='fdr_bh', alpha=alpha)
    result_df = pd.DataFrame({
        'feature':     feats_valid,
        'p_value':     p_arr,
        'q_value':     q_arr,
        'significant': reject,
    }).sort_values('q_value').reset_index(drop=True)
    sig_feats = result_df.loc[result_df['significant'], 'feature'].tolist()
    return sig_feats, result_df


def vip_select(X_train, y_train, anova_features, threshold=1.0):
    """ANOVA pre-filter → PLS-DA VIP ≥ threshold. Train only."""
    X_in    = X_train[anova_features] if len(anova_features) >= 2 else X_train
    classes = sorted(np.unique(y_train))
    y_oh    = np.zeros((len(y_train), len(classes)))
    for i, c in enumerate(classes):
        y_oh[y_train == c, i] = 1.0
    n_comp = min(2, X_in.shape[1], len(classes) - 1)
    pls    = PLSRegression(n_components=n_comp, scale=False)
    pls.fit(X_in.values, y_oh)
    t, w, q_ = pls.x_scores_, pls.x_weights_, pls.y_loadings_
    ss        = np.sum(q_**2, axis=0) * np.sum(t**2, axis=0)
    vip_scores = np.sqrt(X_in.shape[1] * np.sum(w**2 * ss, axis=1) / np.sum(ss))
    vip_df = pd.DataFrame({
        'feature':   X_in.columns,
        'vip_score': vip_scores,
    }).sort_values('vip_score', ascending=False).reset_index(drop=True)
    selected = vip_df.loc[vip_df['vip_score'] >= threshold, 'feature'].tolist()
    return selected, vip_df, pls


def fold_change_top10(df_train, feature_cols, top_n=10):
    """Day별 CLR mean의 max − min (원본 방식). Train only."""
    df = df_train.copy()
    if 'Day' not in df.columns:
        df['Day'] = df['Description'].apply(extract_day)

    day_mean = df.groupby('Day')[feature_cols].mean()
    days     = sorted(day_mean.index)

    abs_change = day_mean.max() - day_mean.min()
    direction  = day_mean.loc[days[-1]] - day_mean.loc[days[0]]

    fc_df = pd.DataFrame({
        'feature':    feature_cols,
        'abs_change': abs_change.values,
        'direction':  direction.values,
    }).sort_values('abs_change', ascending=False).reset_index(drop=True)

    return fc_df['feature'].iloc[:top_n].tolist(), fc_df

In [3]:
clf_configs = {
    "RandomForest": {
        "estimator": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200],
            "max_depth":    [3, 5, 7],
            "min_samples_split": [2, 5],
        },
    },
    "XGBoost": {
        "estimator": XGBClassifier(random_state=42, eval_metric='mlogloss',
                                   verbosity=0, use_label_encoder=False),
        "params": {
            "n_estimators":  [100, 200],
            "max_depth":     [3, 5, 7],
            "learning_rate": [0.05, 0.1],
        },
    },
    "SVC": {
        "estimator": Pipeline([('sc', StandardScaler()),
                               ('svc', SVC(probability=True, class_weight='balanced'))]),
        "params": {
            "svc__C":      [0.5, 1, 2],
            "svc__gamma":  ["scale", "auto"],
            "svc__kernel": ["rbf"],
        },
    },
    "LogisticRegression": {
        "estimator": Pipeline([('sc', StandardScaler()),
                               ('lr', LogisticRegression(penalty='elasticnet', solver='saga',
                                                         max_iter=500, class_weight='balanced',
                                                         random_state=42))]),
        "params": {
            "lr__l1_ratio": [0.2, 0.5, 0.8],
            "lr__C":        [0.5, 1, 2],
        },
    },
    "KNN": {
        "estimator": KNeighborsClassifier(),
        "params": {
            "n_neighbors": [3, 5, 7],
            "weights":     ["uniform", "distance"],
        },
    },
}

In [4]:
class LSTMRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, units=32, dropout=0.2, epochs=100,
                 batch_size=16, learning_rate=0.001, random_state=42):
        self.units         = units
        self.dropout       = dropout
        self.epochs        = epochs
        self.batch_size    = batch_size
        self.learning_rate = learning_rate
        self.random_state  = random_state

    def _build(self, n_feat):
        tf.random.set_seed(self.random_state)
        m = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(1, n_feat)),
            tf.keras.layers.LSTM(self.units, return_sequences=False),
            tf.keras.layers.Dropout(self.dropout),
            tf.keras.layers.Dense(16, activation='relu'),
            tf.keras.layers.Dense(1),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(self.learning_rate), loss='mae')
        return m

    def fit(self, X, y):
        np.random.seed(self.random_state)
        tf.random.set_seed(self.random_state)
        self.n_features_ = X.shape[1]
        self.model_ = self._build(self.n_features_)
        self.model_.fit(X.reshape(-1, 1, self.n_features_), y,
                        epochs=self.epochs, batch_size=self.batch_size, verbose=0)
        return self

    def predict(self, X):
        return self.model_.predict(
            X.reshape(-1, 1, X.shape[1]), verbose=0).flatten()

reg_configs = {
    "LightGBM": {
        "estimator": LGBMRegressor(random_state=42, verbose=-1),
        "params": {
            "n_estimators":  [100, 200],
            "max_depth":     [3, 5, 7],
            "learning_rate": [0.05, 0.1],
            "num_leaves":    [20, 31],
        },
    },
    "RandomForest": {
        "estimator": RandomForestRegressor(random_state=42),
        "params": {
            "n_estimators":      [100, 200],
            "max_depth":         [3, 5, 7],
            "min_samples_split": [2, 5],
        },
    },
    "XGBoost": {
        "estimator": XGBRegressor(random_state=42, eval_metric='mae', verbosity=0),
        "params": {
            "n_estimators":  [100, 200],
            "max_depth":     [3, 5, 7],
            "learning_rate": [0.05, 0.1],
        },
    },
    "SVR": {
        "estimator": Pipeline([('sc', StandardScaler()), ('svr', SVR())]),
        "params": {
            "svr__C":      [0.1, 1, 10],
            "svr__gamma":  ["scale", "auto"],
            "svr__kernel": ["rbf", "linear"],
        },
    },
}

reg_configs['LSTM'] = {
    'estimator': LSTMRegressor(random_state=42),
    'params': {
        'units':         [32, 64],
        'dropout':       [0.1, 0.2],
        'epochs':        [100, 150],
        'learning_rate': [0.001, 0.005],
    },
}

## 2. Data load

- `labeled_splits.pkl` / `kmeans_labeled_splits.pkl` — train/test/validation partitions with the seed-42 fixed-threshold labels
- `threshold_log.pkl` / `kmeans_threshold_log.pkl` — Optuna and K-means thresholds fitted independently on each seed's training set
- `screening_results.pkl`, `classification_results.pkl`, `regression_results.pkl` — seed-42 fixed-threshold results (main pipeline)
- `sensitivity_results.pkl`, `reg_sensitivity_results.pkl` — used for a consistency check in Section 6

In [5]:
with open(f"{PKL_DIR}/labeled_splits.pkl", 'rb') as f:
    optuna_data = pickle.load(f)
with open(f"{PKL_DIR}/kmeans_labeled_splits.pkl", 'rb') as f:
    kmeans_data = pickle.load(f)
with open(f"{PKL_DIR}/threshold_log.pkl", 'rb') as f:
    optuna_thr_log = pickle.load(f)
with open(f"{PKL_DIR}/kmeans_threshold_log.pkl", 'rb') as f:
    kmeans_thr_log = pickle.load(f)
with open(f"{PKL_DIR}/screening_results.pkl", 'rb') as f:
    screen = pickle.load(f)
with open(f"{PKL_DIR}/classification_results.pkl", 'rb') as f:
    fixed_clf = pickle.load(f)
with open(f"{PKL_DIR}/regression_results.pkl", 'rb') as f:
    fixed_reg = pickle.load(f)
with open(f"{PKL_DIR}/sensitivity_results.pkl", 'rb') as f:
    sens_clf = pickle.load(f)
with open(f"{PKL_DIR}/reg_sensitivity_results.pkl", 'rb') as f:
    sens_reg = pickle.load(f)

feature_cols = optuna_data['feature_cols']
tpc_col      = optuna_data['tpc_col']
fixed_splits = {'optuna': optuna_data['labeled_splits'],
                'kmeans': kmeans_data['labeled_splits']}
thr_logs     = {'optuna': optuna_thr_log, 'kmeans': kmeans_thr_log}

thr_rows = []
for seed in SEEDS:
    row = {'seed': seed}
    for lab in ('optuna', 'kmeans'):
        row[f'{lab}_low_fixed']    = thr_logs[lab][LABEL_SEED]['tpc_thr_low']
        row[f'{lab}_high_fixed']   = thr_logs[lab][LABEL_SEED]['tpc_thr_high']
        row[f'{lab}_low_perseed']  = thr_logs[lab][seed]['tpc_thr_low']
        row[f'{lab}_high_perseed'] = thr_logs[lab][seed]['tpc_thr_high']
    thr_rows.append(row)
df_thresholds = pd.DataFrame(thr_rows)

print(f"features: {len(feature_cols)} | tpc_col: {tpc_col}")
print()
print("=== Thresholds (log CFU/g): seed-42 fixed vs per-seed ===")
print(df_thresholds.round(4).to_string(index=False))

features: 110 | tpc_col: TPC (log CFU/g)

=== Thresholds (log CFU/g): seed-42 fixed vs per-seed ===
 seed  optuna_low_fixed  optuna_high_fixed  optuna_low_perseed  optuna_high_perseed  kmeans_low_fixed  kmeans_high_fixed  kmeans_low_perseed  kmeans_high_perseed
   38            4.2214             6.3727              4.2407               6.3515            3.8643             5.2693              3.8362               5.2321
   39            4.2214             6.3727              4.2035               6.2741            3.8643             5.2693              3.8801               5.2567
   40            4.2214             6.3727              4.2289               6.3514            3.8643             5.2693              3.8829               5.3077
   41            4.2214             6.3727              4.3568               6.4225            3.8643             5.2693              3.8070               5.2266
   42            4.2214             6.3727              4.2214               6.3727       

## 3. Check — copied screening functions reproduce `screening_results.pkl`

Feature selection is repeated with the **seed-42 fixed-threshold labels** for all 4 conditions × 10 seeds and compared with the stored feature lists.
The notebook stops here if any list differs, because the per-seed results would then not be comparable with the main pipeline.

In [6]:
def select_features(df_train, cond):
    """3. screening.ipynb 와 같은 방식으로 condition 별 feature 를 선택한다."""
    if cond in ('cond1', 'cond3'):
        feats, _ = fold_change_top10(df_train, feature_cols)
        return feats
    anova_feats, _ = anova_fdr(df_train, feature_cols)
    feats, _, _ = vip_select(df_train[feature_cols], df_train['Label'].values, anova_feats)
    return feats


chk_rows = []
for cond in CONDS:
    for seed in SEEDS:
        tr    = fixed_splits[COND_LABELING[cond]][seed]['train']
        feats = select_features(tr, cond)
        ref   = screen[COND_SCREEN_KEY[cond]]['per_seed_features'][seed]
        chk_rows.append({'condition': cond, 'seed': seed, 'n_features': len(feats),
                         'identical': list(feats) == list(ref)})
df_chk_screen = pd.DataFrame(chk_rows)

print(df_chk_screen.groupby('condition')['identical'].agg(['sum', 'count'])
      .rename(columns={'sum': 'identical', 'count': 'seeds'}).to_string())
assert df_chk_screen['identical'].all(), "복사한 screening 함수가 screening_results.pkl 을 재현하지 못함 — 중단"
print("\n→ 4 conditions × 10 seeds 전부 screening_results.pkl 과 동일")

           identical  seeds
condition                  
cond1             10     10
cond2             10     10
cond3             10     10
cond4             10     10

→ 4 conditions × 10 seeds 전부 screening_results.pkl 과 동일


## 4. Per-seed labels and feature selection

Each seed's train/test/validation samples are relabeled with that seed's own Optuna or K-means thresholds.
Features are then selected on the relabeled training set:

- Conditions 1 and 3 (fold-change Top10): computed from day-wise CLR means, so the selected genera are expected to be identical to the main pipeline.
- Conditions 2 and 4 (VIP ≥ 1.0): PLS-DA uses the labels, so the selected genera can change.

In [7]:
def relabel(seed, labeling):
    """seed 자신의 임계값으로 train/test/val 을 다시 라벨링한다."""
    lo = thr_logs[labeling][seed]['tpc_thr_low']
    hi = thr_logs[labeling][seed]['tpc_thr_high']
    out = {}
    for part in ('train', 'test', 'val'):
        d = fixed_splits[labeling][seed][part].copy()
        d['Label'] = label_by_tpc(d, lo, hi)
        out[part] = d
    return out


perseed_splits = {lab: {seed: relabel(seed, lab) for seed in SEEDS}
                  for lab in ('optuna', 'kmeans')}

# ── 라벨이 바뀐 시료 수 ─────────────────────────────────────────
lab_rows = []
for lab in ('optuna', 'kmeans'):
    for seed in SEEDS:
        row = {'labeling': lab, 'seed': seed}
        for part in ('train', 'test', 'val'):
            a = fixed_splits[lab][seed][part]['Label'].values
            b = perseed_splits[lab][seed][part]['Label'].values
            row[f'n_{part}'] = len(a)
            row[f'n_changed_{part}'] = int((a != b).sum())
        lab_rows.append(row)
df_label_change = pd.DataFrame(lab_rows)

print("=== Samples whose label changes under the per-seed threshold ===")
print(df_label_change.to_string(index=False))

# ── Feature selection ───────────────────────────────────────────
perseed_features = {cond: {} for cond in CONDS}
feat_rows = []
for cond in CONDS:
    for seed in SEEDS:
        tr    = perseed_splits[COND_LABELING[cond]][seed]['train']
        feats = select_features(tr, cond)
        perseed_features[cond][seed] = feats

        ref    = screen[COND_SCREEN_KEY[cond]]['per_seed_features'][seed]
        shared = set(feats) & set(ref)
        feat_rows.append({
            'condition': cond, 'seed': seed,
            'n_fixed': len(ref), 'n_perseed': len(feats), 'n_shared': len(shared),
            'jaccard': len(shared) / len(set(feats) | set(ref)),
            'identical_set': set(feats) == set(ref),
            'only_fixed':   ', '.join(sorted(g.replace('g__', '') for g in set(ref) - set(feats))),
            'only_perseed': ', '.join(sorted(g.replace('g__', '') for g in set(feats) - set(ref))),
        })
df_features = pd.DataFrame(feat_rows)

print()
print("=== Feature selection: seed-42 fixed vs per-seed threshold ===")
print(df_features.groupby('condition')
      .agg(identical_seeds=('identical_set', 'sum'),
           mean_jaccard=('jaccard', 'mean'),
           n_fixed_range=('n_fixed', lambda x: f"{x.min()}-{x.max()}"),
           n_perseed_range=('n_perseed', lambda x: f"{x.min()}-{x.max()}"))
      .round(3).to_string())

=== Samples whose label changes under the per-seed threshold ===
labeling  seed  n_train  n_changed_train  n_test  n_changed_test  n_val  n_changed_val
  optuna    38      189                3      54               0     27              0
  optuna    39      189                6      54               1     27              1
  optuna    40      189                1      54               0     27              0
  optuna    41      189               11      54               4     27              4
  optuna    42      189                0      54               0     27              0
  optuna    43      189                3      54               0     27              0
  optuna    44      189                8      54               2     27              0
  optuna    45      189               14      54               4     27              1
  optuna    46      189                5      54               1     27              0
  optuna    47      189                4      54               0 

## 5. Per-seed threshold models

Same training procedure as the main pipeline (GridSearchCV, cv = 5, same grids and random states):
classification as in `4. classification.ipynb` Section 3, regression (Early + Intermediate samples only) as in `5. regression.ipynb` Section 3.

Results are saved to `per_seed_threshold_results.pkl` after each block, so that Sections 6–7 can be re-run without retraining.

In [8]:
PERSEED_PKL = f"{PKL_DIR}/per_seed_threshold_results.pkl"
per_seed_threshold_results = {'thresholds': df_thresholds,
                              'label_change': df_label_change,
                              'features': perseed_features}


def _save_progress():
    with open(PERSEED_PKL, 'wb') as f:
        pickle.dump(per_seed_threshold_results, f)
    print(f"Saved → {PERSEED_PKL}")


# ══════════════════ Classification ══════════════════
perseed_clf = {c: {m: {} for m in clf_configs} for c in CONDS}

for cond in CONDS:
    sp_all = perseed_splits[COND_LABELING[cond]]
    print(f"\n{'='*60}\nCondition: {COND_LABELS[cond]}\n{'='*60}")
    for seed in SEEDS:
        feats = perseed_features[cond][seed]
        tr, te = sp_all[seed]['train'], sp_all[seed]['test']
        X_train, y_train = tr[feats].values, tr['Label'].values
        X_test,  y_test  = te[feats].values, te['Label'].values

        for model_name, cfg in clf_configs.items():
            grid = GridSearchCV(_base.clone(cfg['estimator']), cfg['params'], cv=5,
                                scoring='f1_macro', n_jobs=-1, refit=True)
            grid.fit(X_train, y_train)
            best   = grid.best_estimator_
            y_pred = best.predict(X_test)
            rec = {'test_acc': accuracy_score(y_test, y_pred),
                   'test_f1':  f1_score(y_test, y_pred, average='macro'),
                   'best_params': grid.best_params_,
                   'features': feats}
            perseed_clf[cond][model_name][seed] = rec

        print(f"  seed={seed} | " + " | ".join(
            f"{m}(acc={perseed_clf[cond][m][seed]['test_acc']:.2f} "
            f"f1={perseed_clf[cond][m][seed]['test_f1']:.2f})" for m in clf_configs))

per_seed_threshold_results['classification'] = perseed_clf
_save_progress()


Condition: Optuna × FC Top10
  seed=38 | RandomForest(acc=0.87 f1=0.87) | XGBoost(acc=0.81 f1=0.79) | SVC(acc=0.87 f1=0.85) | LogisticRegression(acc=0.81 f1=0.78) | KNN(acc=0.83 f1=0.82)
  seed=39 | RandomForest(acc=0.76 f1=0.69) | XGBoost(acc=0.76 f1=0.67) | SVC(acc=0.87 f1=0.86) | LogisticRegression(acc=0.78 f1=0.71) | KNN(acc=0.85 f1=0.83)
  seed=40 | RandomForest(acc=0.93 f1=0.93) | XGBoost(acc=0.87 f1=0.88) | SVC(acc=0.91 f1=0.91) | LogisticRegression(acc=0.85 f1=0.85) | KNN(acc=0.83 f1=0.86)
  seed=41 | RandomForest(acc=0.87 f1=0.87) | XGBoost(acc=0.83 f1=0.84) | SVC(acc=0.85 f1=0.86) | LogisticRegression(acc=0.87 f1=0.84) | KNN(acc=0.87 f1=0.87)
  seed=42 | RandomForest(acc=0.85 f1=0.87) | XGBoost(acc=0.76 f1=0.74) | SVC(acc=0.81 f1=0.80) | LogisticRegression(acc=0.74 f1=0.72) | KNN(acc=0.85 f1=0.85)
  seed=43 | RandomForest(acc=0.81 f1=0.77) | XGBoost(acc=0.81 f1=0.75) | SVC(acc=0.81 f1=0.77) | LogisticRegression(acc=0.80 f1=0.72) | KNN(acc=0.78 f1=0.73)
  seed=44 | RandomFore

In [9]:
# ══════════════════ Regression (Early + Intermediate only) ══════════════════
perseed_reg = {c: {m: {} for m in reg_configs} for c in CONDS}

for cond in CONDS:
    sp_all = perseed_splits[COND_LABELING[cond]]
    print(f"\n{'='*60}\nCondition: {COND_LABELS[cond]}\n{'='*60}")
    for seed in SEEDS:
        feats = perseed_features[cond][seed]
        tr, te = sp_all[seed]['train'], sp_all[seed]['test']
        tr_fm = tr[tr['Label'] != 1]
        te_fm = te[te['Label'] != 1]
        X_train, y_train = tr_fm[feats].values, tr_fm[tpc_col].values
        X_test,  y_test  = te_fm[feats].values, te_fm[tpc_col].values

        for model_name, cfg in reg_configs.items():
            grid = GridSearchCV(_base.clone(cfg['estimator']), cfg['params'], cv=5,
                                scoring='neg_mean_absolute_error', n_jobs=-1, refit=True)
            grid.fit(X_train, y_train)
            best   = grid.best_estimator_
            y_pred = best.predict(X_test)
            rec = {'test_r2':  r2_score(y_test, y_pred),
                   'test_mae': mean_absolute_error(y_test, y_pred),
                   'best_params': grid.best_params_,
                   'features': feats}
            perseed_reg[cond][model_name][seed] = rec

        print(f"  seed={seed} | " + " | ".join(
            f"{m} R²={perseed_reg[cond][m][seed]['test_r2']:.3f}" for m in reg_configs))

per_seed_threshold_results['regression'] = perseed_reg
_save_progress()


Condition: Optuna × FC Top10


I0000 00:00:1794860967.926087 1399173 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794860967.926672 1399169 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794860967.931687 1399164 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794860967.937277 1399157 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

  seed=38 | LightGBM R²=0.706 | RandomForest R²=0.748 | XGBoost R²=0.683 | SVR R²=0.620 | LSTM R²=0.647


I0000 00:00:1794861018.485124 2084986 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794861018.500544 2088780 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794861018.500543 2015350 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794861018.510961 2090685 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

  seed=39 | LightGBM R²=0.684 | RandomForest R²=0.680 | XGBoost R²=0.725 | SVR R²=0.606 | LSTM R²=0.654


I0000 00:00:1794861064.616356 2793391 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794861064.646586 2759296 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794861064.687382 2793389 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794861064.718173 2685358 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

... (71 outputs left out)

I0000 00:00:1794862795.524784 2753162 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794862795.558719 2726668 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794862795.569745 2785384 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794862795.590451 2779487 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

  seed=46 | LightGBM R²=0.437 | RandomForest R²=0.438 | XGBoost R²=0.386 | SVR R²=0.341 | LSTM R²=0.367


I0000 00:00:1794862841.392070 3371723 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794862841.424485 3486148 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794862841.427156 3486150 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794862841.438738 3478428 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

  seed=47 | LightGBM R²=0.406 | RandomForest R²=0.419 | XGBoost R²=0.258 | SVR R²=0.406 | LSTM R²=0.352
Saved → output/pkl/per_seed_threshold_results.pkl


## 6. Consistency checks

1. **Condition 1 × RF vs the sensitivity analysis.** `sensitivity_results.pkl` / `reg_sensitivity_results.pkl` (label seed = train seed) contain the same per-seed-threshold models for Condition 1. The values should be identical.
2. **Seed 42.** For seed 42 the per-seed thresholds equal the fixed thresholds, so labels, features and models should be identical to the main pipeline for every condition and model. LSTM can differ slightly because of floating-point non-determinism.

In [10]:
chk_rows = []

# (1) Cond 1 × RF vs sensitivity diagonal
for task, res, sens, keys in (('Classification', perseed_clf, sens_clf, (('test_acc', 'acc'), ('test_f1', 'f1'))),
                              ('Regression',     perseed_reg, sens_reg, (('test_r2', 'r2'), ('test_mae', 'mae')))):
    for k_new, k_ref in keys:
        d = max(abs(res[FINAL_COND][FINAL_MODEL][s][k_new] - sens[FINAL_MODEL][s][s][k_ref]) for s in SEEDS)
        chk_rows.append({'check': f'{task}: Cond 1 × RF vs sensitivity [seed][seed]',
                         'condition': FINAL_COND, 'model': FINAL_MODEL, 'metric': k_new,
                         'max_abs_diff': d})

# (2) seed 42: per-seed vs fixed
for task, res, fixed, cfgs, keys in (('Classification', perseed_clf, fixed_clf, clf_configs, ('test_acc', 'test_f1')),
                                     ('Regression',     perseed_reg, fixed_reg, reg_configs, ('test_r2', 'test_mae'))):
    for cond in CONDS:
        for m in cfgs:
            for k in keys:
                chk_rows.append({'check': f'{task}: seed 42 per-seed vs fixed',
                                 'condition': cond, 'model': m, 'metric': k,
                                 'max_abs_diff': abs(res[cond][m][LABEL_SEED][k] - fixed[cond][m][LABEL_SEED][k])})

df_checks = pd.DataFrame(chk_rows)
df_checks['status'] = np.where(df_checks['max_abs_diff'] < 1e-12, 'OK',
                               np.where(df_checks['model'] == 'LSTM', 'differs (LSTM)', 'MISMATCH'))
print(df_checks.groupby(['check', 'status']).size().to_string())
if (df_checks['status'] == 'MISMATCH').any():
    print("\n[MISMATCH]")
    print(df_checks[df_checks['status'] == 'MISMATCH'].to_string(index=False))
if (df_checks['status'] == 'differs (LSTM)').any():
    print("\n[LSTM]")
    print(df_checks[df_checks['status'] == 'differs (LSTM)'].to_string(index=False))

check                                                    status        
Classification: Cond 1 × RF vs sensitivity [seed][seed]  OK                 2
Classification: seed 42 per-seed vs fixed                OK                40
Regression: Cond 1 × RF vs sensitivity [seed][seed]      OK                 2
Regression: seed 42 per-seed vs fixed                    OK                32
                                                         differs (LSTM)     8

[LSTM]
                                check condition model   metric  max_abs_diff         status
Regression: seed 42 per-seed vs fixed     cond1  LSTM  test_r2      0.094796 differs (LSTM)
Regression: seed 42 per-seed vs fixed     cond1  LSTM test_mae      0.058179 differs (LSTM)
Regression: seed 42 per-seed vs fixed     cond2  LSTM  test_r2      0.005262 differs (LSTM)
Regression: seed 42 per-seed vs fixed     cond2  LSTM test_mae      0.005805 differs (LSTM)
Regression: seed 42 per-seed vs fixed     cond3  LSTM  test_r2      0.

## 7. CV test performance — seed-42 fixed vs per-seed threshold

All 4 conditions × 5 models, classification and regression. Mean ± SD over the 10 seeds, paired t-test with seeds as pairs.

The best model per condition is selected as in the main pipeline (classification: highest mean test macro-F1; regression: highest mean test R²).

In [11]:
perf_rows, raw_rows = [], []
for task, fixed, per, cfgs, metrics in (
        ('Classification', fixed_clf, perseed_clf, clf_configs,
         (('Accuracy', 'test_acc'), ('Macro-F1', 'test_f1'))),
        ('Regression', fixed_reg, perseed_reg, reg_configs,
         (('R2', 'test_r2'), ('MAE (log CFU/g)', 'test_mae')))):
    for cond in CONDS:
        for m in cfgs:
            for metric, key in metrics:
                a = np.array([fixed[cond][m][s][key] for s in SEEDS])
                b = np.array([per[cond][m][s][key]   for s in SEEDS])
                t, p = stats.ttest_rel(a, b)
                perf_rows.append({'task': task, 'condition': cond,
                                  'condition_label': COND_LABELS[cond], 'model': m,
                                  'metric': metric,
                                  'seed42_fixed_mean': a.mean(), 'seed42_fixed_sd': a.std(),
                                  'per_seed_mean':     b.mean(), 'per_seed_sd':     b.std(),
                                  'delta': b.mean() - a.mean(), 't': t, 'p_paired': p,
                                  'significant_a0.05': 'yes' if p < 0.05 else 'no'})
                for s, va, vb in zip(SEEDS, a, b):
                    raw_rows.append({'task': task, 'condition': cond, 'model': m,
                                     'metric': metric, 'seed': s,
                                     'seed42_fixed': va, 'per_seed': vb})
df_perf = pd.DataFrame(perf_rows)
df_perf_raw = pd.DataFrame(raw_rows)

# ── condition 별 best model ─────────────────────────────────────
best_rows = []
for task, fixed, per, cfgs, key in (('Classification', fixed_clf, perseed_clf, clf_configs, 'test_f1'),
                                    ('Regression',     fixed_reg, perseed_reg, reg_configs, 'test_r2')):
    for cond in CONDS:
        row = {'task': task, 'condition': cond, 'condition_label': COND_LABELS[cond]}
        for scheme, res in (('seed42_fixed', fixed), ('per_seed', per)):
            means = {m: np.mean([res[cond][m][s][key] for s in SEEDS]) for m in cfgs}
            best  = max(means, key=means.get)
            row[f'{scheme}_best_model'] = best
            row[f'{scheme}_best_mean_{key}'] = means[best]
        row['same_best_model'] = row['seed42_fixed_best_model'] == row['per_seed_best_model']
        best_rows.append(row)
df_best = pd.DataFrame(best_rows)

pd.set_option('display.width', 200)
for task in ('Classification', 'Regression'):
    print(f"\n=== {task}: seed-42 fixed vs per-seed threshold (CV test, 10 seeds) ===")
    print(df_perf[df_perf['task'] == task]
          .drop(columns=['task', 'condition_label']).round(4).to_string(index=False))

print("\n=== Best model per condition ===")
print(df_best.round(4).to_string(index=False))


=== Classification: seed-42 fixed vs per-seed threshold (CV test, 10 seeds) ===
condition              model   metric  seed42_fixed_mean  seed42_fixed_sd  per_seed_mean  per_seed_sd   delta       t  p_paired significant_a0.05
    cond1       RandomForest Accuracy             0.8574           0.0275         0.8481       0.0437 -0.0093  0.7454    0.4751                no
    cond1       RandomForest Macro-F1             0.8476           0.0419         0.8331       0.0647 -0.0145  0.9186    0.3823                no
    cond1            XGBoost Accuracy             0.8074           0.0323         0.8204       0.0352  0.0130 -1.1372    0.2848                no
    cond1            XGBoost Macro-F1             0.7884           0.0486         0.7956       0.0576  0.0072 -0.7267    0.4859                no
    cond1                SVC Accuracy             0.8463           0.0406         0.8519       0.0341  0.0056 -0.8955    0.3938                no
    cond1                SVC Macro-F1      

## 8. Save

In [12]:
per_seed_threshold_results.update({
    'checks_screening': df_chk_screen, 'checks_models': df_checks,
    'performance': df_perf, 'best_model_by_condition': df_best,
})
_save_progress()

xlsx = f"{OUTPUT_DIR}/per_seed_threshold_test.xlsx"
with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
    df_thresholds.to_excel(w,   sheet_name='Thresholds',               index=False)
    df_label_change.to_excel(w, sheet_name='Label_change',             index=False)
    df_features.to_excel(w,     sheet_name='Feature_selection',        index=False)
    df_perf[df_perf['task'] == 'Classification'].to_excel(w, sheet_name='Perf_classification', index=False)
    df_perf[df_perf['task'] == 'Regression'].to_excel(w,     sheet_name='Perf_regression',     index=False)
    df_best.to_excel(w,         sheet_name='Best_model_by_condition',  index=False)
    df_perf_raw.to_excel(w,     sheet_name='Perf_per_seed_raw',        index=False)
    pd.concat([df_chk_screen.assign(check='screening reproduces screening_results.pkl'),
               df_checks], ignore_index=True).to_excel(w, sheet_name='Checks', index=False)
print(f"Excel saved → {xlsx}")

Saved → output/pkl/per_seed_threshold_results.pkl
Excel saved → output/per_seed_threshold_test.xlsx
